# 통합수업일지 데이터 분석기법 실습관

**모든 데이터는 교육용 가상 자료**이며 실제 스포츠센터·영화 이용자 기록이 아닙니다.

**근거:** 2026-10-29(Pandas·결측), 10-30(통계검정·회귀·군집), 11-02(groupby·corr·t-test·StandardScaler·K-Means·Elbow·영화추천), 11-03(분석기법 지도), 11-09(시각화·GitHub 제출). [원본 통합일지](https://docs.google.com/document/d/1vSeKZW3o3qWyyTn_n1IAkFmWvDxNO4j2cfKaMQaIZtk/edit)

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
df = pd.read_csv(ROOT/'data/sports_center_members_demo.csv')
print('원본 행',len(df),'중복 회원 ID',df.member_id.duplicated().sum())
display(df.head())
display(df.describe(include='all'))

## 1. 전처리: 결측치·중복·이상값

회원 ID 중복 2건을 삭제하고, 음수 방문 1건은 결측으로 바꾼 후 각 열의 중앙값으로 채웁니다. 이는 실습용 대치 방식이며, 실제 분석에서는 누락 원인을 먼저 확인해야 합니다.

In [ ]:
d = df.drop_duplicates('member_id').copy()
d.loc[d.visits_per_month<0,'visits_per_month'] = np.nan
missing = d[['monthly_fee_krw_1000','visits_per_month','satisfaction_1to5']].isna().sum()
print('대치 전 결측:',missing.to_dict())
for col in missing.index:
    d[col] = d[col].fillna(d[col].median())
assert len(d)==48 and not d.isna().any().any()
print('대치 후 행:',len(d))

## 2. 묶음 계산(groupby, agg)과 표 병합(merge)

프로그램 코드 A/B를 설명 표와 병합해 회원 수와 월평균 방문 횟수를 계산합니다.

In [ ]:
meta = pd.read_csv(ROOT/'data/program_metadata_demo.csv')
joined = d.merge(meta,on='program',validate='many_to_one')
group = joined.groupby(['program','program_name']).agg(n=('member_id','size'),avg_visits=('visits_per_month','mean'))
display(group)

## 3. 상관관계와 t-검정

상관계수 r은 두 변수의 함께 움직이는 방향과 강도, p값은 특정 귀무가설 아래 관측 데이터의 극단성을 설명합니다. 둘 다 원인 관계를 증명하지 않습니다.

In [ ]:
from scipy.stats import ttest_ind
print('가입개월-방문 상관계수:',round(d.months_active.corr(d.visits_per_month),4))
a = d.loc[d.program=='A','visits_per_month']
b = d.loc[d.program=='B','visits_per_month']
t,p = ttest_ind(a,b,equal_var=False)
print('Welch t:',round(float(t),4),'p:',round(float(p),6))

## 4. 회귀와 학습/테스트 분리

연령, 활동 개월, 월 요금, 프로그램 구분으로 방문 횟수를 예측합니다. **훈련 데이터에서만** 결측 대치와 표준화 규칙을 학습합니다. 성능이 나쁘면 그 사실도 그대로 보고합니다.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
X = d[['age','months_active','monthly_fee_krw_1000']].copy()
X['program_B'] = (d.program=='B').astype(int)
xt,xv,yt,yv = train_test_split(X,d.visits_per_month,test_size=.25,random_state=42)
model = make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),LinearRegression())
model.fit(xt,yt)
pred = model.predict(xv)
print('훈련/테스트:',len(xt),len(xv),'MAE:',round(mean_absolute_error(yv,pred),4),'R²:',round(r2_score(yv,pred),4))

## 5. 표준화, 군집, Elbow

요금·방문수·만족도의 단위가 달라 StandardScaler로 변환 후 K-Means를 적용합니다. k=2~6의 SSE를 비교하되 Elbow는 군집 개수의 자동 정답이 아닙니다.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
z = StandardScaler().fit_transform(d[['visits_per_month','monthly_fee_krw_1000','satisfaction_1to5']])
errors={k:round(KMeans(n_clusters=k,n_init=20,random_state=42).fit(z).inertia_,4) for k in range(2,7)}
cluster=KMeans(n_clusters=3,n_init=20,random_state=42).fit_predict(z)
print('Elbow SSE:',errors)
print('군집 크기:',pd.Series(cluster).value_counts().sort_index().to_dict())
print('실루엣:',round(silhouette_score(z,cluster),4))

## 6. 영화추천: 코사인 유사도와 KNN

영화별 5명 평점을 벡터로 보고, 영화A와 가장 유사한 작품을 찾습니다. 0점은 평가하지 않았다는 교육용 약속입니다.

In [ ]:
from sklearn.neighbors import NearestNeighbors
movies=pd.read_csv(ROOT/'data/movie_ratings_demo.csv')
ratings=movies.drop(columns='movie').to_numpy()
nn=NearestNeighbors(metric='cosine',n_neighbors=4).fit(ratings)
dist,ids=nn.kneighbors(ratings[[0]])
for movie_id,distance in zip(ids[0],dist[0]):
    if movie_id != 0:
        print(movies.loc[movie_id,'movie'],'코사인 유사도',round(1-float(distance),4))

## 7. 동일 분석 재현과 기록

GitHub 저장소의 `analysis.py`는 위 과정을 처음부터 실행해 `data/analysis_results.json`을 재작성합니다. 화면은 저장된 JSON 결과를 보여주는 **정적 교육용 웹페이지**입니다. 입력값은 실제 통계로 사용하지 마세요.

In [ ]:
import json
result=json.loads((ROOT/'data/analysis_results.json').read_text(encoding='utf-8'))
print('결측',result['missing_counts'],'상관',result['corr'],'p값',result['welch_p'],'회귀 R²',result['regression']['r2'])